In [ ]:

from google.colab import drive
drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/depi_scraping"

Mounted at /content/drive


In [ ]:
import json, re, time
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo
import requests
from bs4 import BeautifulSoup

CONTACT = "your.ayaelgrwany18@gmail.com"
SITE = "jumia"
BASE_URL = "https://www.jumia.com.eg"
MAX_PAGES = 50
DELAY_SECONDS = 4

CATEGORIES = {
    "/mobile-phones/": "phone",
    "/televisions/": "tv",
}

session = requests.Session()
session.headers.update({"User-Agent": f"DEPI-Student-PriceBot/1.0 (educational project; contact: {CONTACT})",
                        "Accept-Language": "en,ar;q=0.8"})
CAIRO = ZoneInfo("Africa/Cairo")
OFFER_FIELDS = ["site", "site_product_id", "snapshot_date", "scraped_at", "url", "title", "brand_raw", "model_raw", "ean",
                "category_site", "category", "sale_price", "original_price", "discount_pct", "currency", "stock_text",
                "in_stock", "seller", "source_format", "raw_file"]


def fetch_html(url, retries=5):
    wait = 2
    for attempt in range(1, retries + 1):
        time.sleep(DELAY_SECONDS)
        try:
            r = session.get(url, timeout=30)
            if r.status_code == 403:
                print("HTTP 403 (blocked) on", url, "- we stop here, we do not work around it")
                return None
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return r.text
        except requests.RequestException as error:
            if attempt == retries:
                raise RuntimeError(f"Gave up on {url}: {error}")
            print("retry", attempt, url, error)
            time.sleep(wait)
            wait *= 2


def parse_price(text):
    if text is None:
        return None
    m = re.search(r"\d[\d,]*(?:\.\d+)?", str(text))
    if not m:
        return None
    v = float(m.group(0).replace(",", ""))
    return v if v > 0 else None


def discount_pct(sale, original):
    if sale and original and original > sale:
        return round(100 * (1 - sale / original))
    return None


def new_offer(**values):
    doc = {f: None for f in OFFER_FIELDS}
    doc["currency"] = "EGP"
    doc.update(values)
    return doc

In [ ]:
                                                                                                                                                                                                                                         def model_code(title):
    codes = [w.upper() for w in re.findall(r"[A-Za-z0-9\-]{6,}", title)
             if re.search(r"\d", w) and re.search(r"[A-Za-z]", w)]
    codes = [c for c in codes if not re.fullmatch(r"\d+(INCH|GB|TB|HZ|P)", c)]
    return max(codes, key=len) if codes else None


def parse_card(card, category_site, snapshot_date, scraped_at, raw_file):
    link = card.select_one("a.core") or card.select_one("a[href]")
    name = card.select_one(".name")
    price = card.select_one(".prc")
    if not link or not name or not price:
        return None
    url = BASE_URL + link["href"] if link["href"].startswith("/") else link["href"]
    sku = link.get("data-ga4-item_id") or link.get("data-id") or card.get("data-sku")
    if not sku:
        m = re.search(r"-(\d+)\.html", url)
        sku = m.group(1) if m else url
    title = name.get_text(strip=True)
    brand = link.get("data-ga4-item_brand") or title.split()[0]
    sale = parse_price(price.get_text())
    old = card.select_one(".old")
    original = parse_price(old.get_text()) if old else None
    if not (original and sale and original > sale):
        original = None
    category = CATEGORIES[category_site]
    return new_offer(
        site=SITE,
        site_product_id=sku,
        snapshot_date=snapshot_date,
        scraped_at=scraped_at,
        url=url,
        title=title,
        brand_raw=brand,
        model_raw=model_code(title) if category == "tv" else None,
        category_site=category_site,
        category=category,
        sale_price=sale,
        original_price=original,
        discount_pct=discount_pct(sale, original),
        in_stock=True,
        source_format="html",
        raw_file=raw_file,
    )

In [ ]:
now = datetime.now(timezone.utc)
snapshot_date = now.astimezone(CAIRO).strftime("%Y-%m-%d")
scraped_at = now.strftime("%Y-%m-%dT%H:%M:%SZ")

offers = []
seen = set()
for path in CATEGORIES:
    for page in range(1, MAX_PAGES + 1):
        html = fetch_html(f"{BASE_URL}{path}?page={page}")
        if html is None:
            break
        cards = BeautifulSoup(html, "html.parser").select("article.prd")
        raw_file = f"raw/{snapshot_date}/jumia/{path.strip('/')}_page{page}.html"
        page_offers = [o for o in (parse_card(c, path, snapshot_date, scraped_at, raw_file) for c in cards) if o]
        ids = {o["site_product_id"] for o in page_offers}
        if not page_offers or ids <= seen:
            break
        offers += [o for o in page_offers if o["site_product_id"] not in seen]
        seen |= ids
        print(path, "page", page, "->", len(page_offers), "offers")
        if len(cards) < 40:
          break

print("total offers:", len(offers))

/mobile-phones/ page 1 -> 40 offers
/mobile-phones/ page 2 -> 40 offers
/mobile-phones/ page 3 -> 40 offers
/mobile-phones/ page 4 -> 40 offers
/mobile-phones/ page 5 -> 40 offers
/mobile-phones/ page 6 -> 40 offers
/mobile-phones/ page 7 -> 40 offers
/mobile-phones/ page 8 -> 40 offers
/mobile-phones/ page 9 -> 40 offers
/mobile-phones/ page 10 -> 40 offers
/mobile-phones/ page 11 -> 40 offers
retry 1 https://www.jumia.com.eg/mobile-phones/?page=12 HTTPSConnectionPool(host='www.jumia.com.eg', port=443): Read timed out. (read timeout=30)
retry 2 https://www.jumia.com.eg/mobile-phones/?page=12 HTTPSConnectionPool(host='www.jumia.com.eg', port=443): Read timed out. (read timeout=30)
/mobile-phones/ page 12 -> 40 offers
retry 1 https://www.jumia.com.eg/mobile-phones/?page=13 HTTPSConnectionPool(host='www.jumia.com.eg', port=443): Read timed out. (read timeout=30)
/mobile-phones/ page 13 -> 40 offers
/mobile-phones/ page 14 -> 40 offers
HTTP 403 (blocked) on https://www.jumia.com.eg/mobile

In [ ]:
p12 = [o for o in offers if o["raw_file"].endswith("televisions_page12.html")]
print(len(p12))
for o in p12[:15]:
    print(o["sale_price"], o["title"][:70])

0


In [ ]:
offers = [o for o in offers if not o["raw_file"].endswith("televisions_page12.html")]
print("total offers:", len(offers))

total offers: 957


In [ ]:
assert offers, "No offers scraped from Jumia - check the site or the parser"
path = Path(OUT_DIR) / f"jumia_offers_{snapshot_date}.json"
path.write_text(json.dumps(offers, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", path)

from collections import Counter
print("per category:", dict(Counter(o["category"] for o in offers)))
print("no price:", sum(1 for o in offers if not o["sale_price"]))
offers[0]

saved: /content/drive/MyDrive/depi_scraping/jumia_offers_2026-10-29.json
per category: {'phone': 558, 'tv': 399}
no price: 0


{'site': 'jumia',
 'site_product_id': 'HO474MP26BRG1NAFAMZ',
 'snapshot_date': '2026-10-29',
 'scraped_at': '2026-10-29T10:58:56Z',
 'url': 'https://www.jumia.com.eg/honor-x5c-plus-dual-sim-4g-128gb4gb-midnight-black-133885513.html',
 'title': 'Honor X5c Plus Dual SIM 4G 128GB/4GB - Midnight Black',
 'brand_raw': 'Honor',
 'model_raw': None,
 'ean': None,
 'category_site': '/mobile-phones/',
 'category': 'phone',
 'sale_price': 8699.0,
 'original_price': 9570.0,
 'discount_pct': 9,
 'currency': 'EGP',
 'stock_text': None,
 'in_stock': True,
 'seller': None,
 'source_format': 'html',
 'raw_file': 'raw/2026-10-29/jumia/mobile-phones_page1.html'}

In [ ]:
import pandas as pd

def tv_key(title):
    codes = re.findall(r"[A-Za-z0-9\-]{4,}", str(title))
    codes = [c.upper().replace("-", "") for c in codes if re.search(r"\d", c) and re.search(r"[A-Za-z]", c)]
    codes = [c for c in codes if c not in {"HDR10", "DVBT2", "1080P", "720P", "2160P"}
             and not re.fullmatch(r"\d+(INCH|K|HZ|W|GB)", c)]
    if not codes:
        return None
    code = max(codes, key=len)
    return code[:-4] if code.endswith("UXEG") else code

rows = []
for site in ["dream2000", "xprs", "jumia"]:
    data = json.loads((Path(OUT_DIR) / f"{site}_offers_{snapshot_date}.json").read_text(encoding="utf-8"))
    rows += [{"site": site, "title": o["title"]} for o in data if o["category"] == "tv"]

tv = pd.DataFrame(rows)
tv["key"] = tv["title"].apply(tv_key)
sites = tv.dropna(subset=["key"]).groupby("key")["site"].nunique()
print("TVs per site:", tv["site"].value_counts().to_dict())
print("TVs on 2+ sites:", (sites >= 2).sum())
print("TVs on 3 sites:", (sites == 3).sum())

TVs per site: {'jumia': 399, 'dream2000': 60, 'xprs': 21}
TVs on 2+ sites: 27
TVs on 3 sites: 2
